# 📘 Notebook 2 — Multilingual Pipeline (English + Spanish)
### GPT‑4o Vision OCR → Embeddings → Auto‑Label (A1) for EN + ES


In [ ]:
!pip install openai pandas tqdm pyarrow requests

In [ ]:
from openai import OpenAI
import pandas as pd, numpy as np, time, os
from tqdm import tqdm
client = OpenAI(api_key='YOUR_OPENAI_KEY')

## Step 1 — Load English + Spanish MultiFinBen

In [ ]:
df_en = pd.read_parquet('ENGLISH_SAS_URL')
df_es = pd.read_parquet('SPANISH_SAS_URL')
df_en['language']='en'; df_es['language']='es'
df = pd.concat([df_en, df_es]).reset_index(drop=True)
df.head()

## Step 2 — GPT‑4o Vision OCR (Multilingual)

In [ ]:
def gpt4o_vision_ocr(b64):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{ 'role':'user','content':[
            {'type':'text','text':'Extract ONLY the text from this image.'},
            {'type':'image_url','image_url':{'url': f'data:image/png;base64,{b64}'}}]}])
    return r.choices[0].message.content

In [ ]:
df['ocr_text']=None
COST=0.005; MAX=8.0; spent=0; count=0
for i in tqdm(range(len(df))):
    if df.loc[i,'ocr_text'] not in [None,'']: continue
    if spent+COST>MAX: break
    try:
        txt=gpt4o_vision_ocr(df.loc[i,'image']); df.loc[i,'ocr_text']=txt
        spent+=COST; count+=1
    except: time.sleep(1)
df.to_pickle('multi_ocr.pkl'); count

## Step 3 — OpenAI Embeddings (Multilingual)

In [ ]:
def embed(t):
    if t is None or len(t.strip())==0: return None
    r=client.embeddings.create(model='text-embedding-3-large',input=t)
    return r.data[0].embedding

In [ ]:
df['embedding']=[embed(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('multi_emb.pkl')

## Step 4 — Auto‑Label Using GPT‑4o (A1 Financial Sections)

In [ ]:
CATS=['Risk','Financials','Management','Operations','Legal','Other']
def auto_label(t):
    if t is None or len(t.strip())==0: return 'Other'
    r=client.chat.completions.create(model='gpt-4o-mini',messages=[
        {'role':'system','content':'Classify into ONE: '+', '.join(CATS)},
        {'role':'user','content':t}])
    out=r.choices[0].message.content.strip()
    return out if out in CATS else 'Other'

In [ ]:
df['label']=[auto_label(t) for t in tqdm(df['ocr_text'])]
df.to_pickle('multi_labeled.pkl'); df['label'].value_counts()

## Save Final Multilingual Dataset

In [ ]:
df.to_pickle('MultiFinBen_EN_ES_ready.pkl')